<!-- THIS IS THE WORKING NOTEBOOK. Edit it here.
     It was first generated from a builder in trench_pull_fluidity_wip, which
     is now a FROZEN ARCHIVE; that header told the reader to edit the copy in
     the archive instead, and has been wrong since this became the working
     and published repository (corrected 2026-10-10). Nothing in this
     repository reads anything from the archive. -->

# Time evolution of the plate-wide force balance

The companion to `01_single_timestep.ipynb`. That notebook derives the
balance at one snapshot from the model output; this one carries the same
quantities across all 37 analysed states of both runs, and reproduces the
content of the paper's time-series figures.

**It needs no model data.** Everything comes from
`notebooks/outputs/column_profiles_deep.npz`, the reduced cache that every
figure script reads: per snapshot, vertical profiles to 250 km of
$\sigma_{zz}$, $\sigma_{xx}$, $\tau_{zx}$, density and temperature at the
trench, first-isostatic, ridge and maximum-moment columns, plus the picks.
That cache is 3 MB; the archive it was reduced from is 1.4 GB.

| from the cache | meaning |
|---|---|
| `t` | model time [Myr], 37 states from 8 to 80 Myr |
| `xT`, `xI`, `xR`, `xM` | trench, first isostatic, ridge and maximum-moment columns |
| `szz_*`, `sxx_*`, `txz_*` | column profiles of the stress components |
| `FB_I`, `FB_R` | basal traction accumulated from the trench |

The spin-up cut is **built into the cache**: states below 8 Myr are
dropped at build time, not masked here, so every figure below inherits it
whether or not it says so. See `docs/ANALYSIS_CONVENTIONS.md` §7.5 for
what that excludes — the whole free-sinking stage, including the fastest
plate motion of the run.

## 1. Setup & imports

In [ ]:
import numpy as np
if not hasattr(np, 'trapz'): np.trapz = np.trapezoid  # numpy 2.x compat
import glob
import natsort
import matplotlib.pyplot as plt
import pyvista as pv
import warnings, logging, gc
from scipy.integrate import cumulative_trapezoid
from scipy.ndimage import gaussian_filter1d
from pathlib import Path

pv.set_error_output_file(Path('/tmp/vtk_errors.log'))
logging.getLogger('pyvista').setLevel(logging.CRITICAL)
logging.getLogger().setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore', message='The VTK reader .*vtkXMLPUnstructuredGridReader.*')

plt.rcParams.update({'mathtext.fontset': 'cm', 'font.family': 'serif'})

## 2. Configuration

Same setup conventions as the other Cerpa notebooks. `MIRROR_X = True` mirrors the model along x so subduction matches MDOODZ orientation.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

_r = os.getcwd()
while not os.path.isdir(os.path.join(_r, "scripts")) and os.path.dirname(_r) != _r:
    _r = os.path.dirname(_r)
sys.path.insert(0, os.path.join(_r, "scripts"))
import column_profiles_cache as cpc

if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

# The integration depth and the spin-up cut are properties of the CACHE,
# not of this notebook: both were applied when it was built. Changing
# either means rebuilding (scripts/column_profiles_cache.py, ~20 min).
print(f"z_c = {cpc.ZC_KM:.0f} km   spin-up cut t >= {cpc.T_MIN_MYR:.0f} Myr")
plt.rcParams["figure.dpi"] = 110

# Anything these cells save goes to notebooks/out/, never to figures/ --
# that directory holds the published figures, made by the scripts.
from pathlib import Path
figures_dir = Path("out"); figures_dir.mkdir(exist_ok=True)

## 3. Load the reduced time series

The working version of this notebook ran its own loop over the archive and
cached the result. That loop is kept below, commented, as the record of how
the cache was built — but it is not what runs here. Re-deriving the series
would give numbers that disagree with the figures beside them: the loop
sampled every other snapshot, applied no spin-up cut, and integrated to the
earlier $z_c$ of 75 km.

Instead the committed cache is loaded and reshaped into the `records`
dictionary the plotting cells below expect, so those cells are unchanged
from the working notebook and their numbers match the paper by
construction.

In [ ]:
import numpy as np

d = cpc.load()

def build_records(key):
    """The cache, reshaped into the `records` dict the figures below use."""
    c = cpc.derive(d, key)
    r = cpc.resultants(d, key)
    p = cpc.partition(d, key)
    z, zc = c['z'], c['zc']
    I = lambda a: np.trapezoid(a[:, zc], z[zc], axis=1)
    gpe = lambda col: -I(d[f'{key}_szz_{col}'])
    gT, gI, gR = gpe('T'), gpe('I'), gpe('R')
    zkm = z / 1e3
    return dict(
        t_yr        = c['t'] * 1e6,
        x_trench    = d[f'{key}_xT'],  xi_loc = d[f'{key}_xI'],
        rloc        = d[f'{key}_xR'],  xm_loc = d[f'{key}_xM'],
        Fd_xT       = r['nd_T'],  nd_I = r['nd_I'],  nd_R = r['nd_R'],
        V_xT        = r['v_T'],   h_np = r['h_np'],
        delta_Fd    = r['nd_I'] - r['nd_T'],
        delta_Fd_R  = r['nd_R'] - r['nd_T'],
        delta_GPE   = gI - gT,
        delta_GPE_R = gR - gT,
        f_tilt      = p['tilt'],
        q_R         = -c['dP'],
        v_sp        = np.abs(d[f'{key}_vx_TR'][:, zkm < 20].mean(axis=1)),
        FB_R        = d[f'{key}_FB_R'],
        delta_sig_zz_zt = d[f'{key}_szz_I'] - d[f'{key}_szz_T'],
        z           = z,
    )

records_STD, records_WAL = build_records('STD'), build_records('WAL')
records = records_STD
print(f"STD {len(records_STD['t_yr'])} states, "
      f"WAL {len(records_WAL['t_yr'])} states, "
      f"{records_STD['t_yr'][0]/1e6:.0f}-{records_STD['t_yr'][-1]/1e6:.0f} Myr")

# --- HOW THE CACHE WAS BUILT (not run here) -----------------------------
# for ff in natsort.natsorted(glob.glob(DATA + f'{KEY}_RefModel/*.pvtu')):
#     v = pv.read(ff)
#     t_myr = float(v['NormalSP::Time'].flat[0]) / 31557600.0 / 1e6
#     if t_myr < 8.0:          # the spin-up cut, applied at BUILD time
#         continue
#     ... sample onto the 1 km grid, pick x_T / x_I / x_R / x_m,
#     ... store the +-5 km column means to 250 km
# See scripts/column_profiles_cache.py for the real thing (~20 min).

## 7. Multi-model figures (from cached records)

Slide-grade figures comparing STD and WAL.  Each cell uses `records_STD`
and `records_WAL` from §6 directly — no per-cell loading.  Output PNGs
go to `figures/` with neutral filenames (no `MODEL_KEY` suffix) since
both models appear in every figure.


**Interpretation**

* $\Delta\mathrm{GPE}^*$ is the **only consistently driving term**, rising from ~2 to ~5 TN/m as the plate ages and its topography grows. $F_B$ is uniformly resisting (~1–3 TN/m).
* $\Delta N_D$ starts near neutral (STD briefly driving early on) and then spends **most of both runs resisting** — increasingly so with time. The normal-stress-difference resultant opposes the plate's motion for the majority of the history: the reverse of the slab-pull reading, and the normal state of these models.

In [ ]:
# Time series of the three plate-wide terms -- ridge-to-trench column
# differences -- for STD and WAL.
#
# ⚠ SIGNS CORRECTED 2026-10-10. This cell plotted -delta_Fd_R under the
# label Delta N_D, and formed the residual as delta_Fd_R - delta_GPE_R,
# which is -F_B. Both were leftovers of the mirrored frame, in which
# driving was -x; the analysis has run in the model's own orientation
# since 2026-09-28, where driving is +x. The comment that justified the
# flip -- 'so the residual sits in the resisting half' -- was describing
# the symptom. Cell 13 below already used delta_Fd_R and the cached FB_R
# unnegated, so the notebook disagreed with itself.
#
# The balance is  Delta N_D - Delta GPE* + F_B = 0,  so F_B = Delta GPE*
# - Delta N_D. Rather than re-derive it, this now plots the CACHED F_B,
# accumulated from tau_zx along the plane at z_c at build time -- which is
# an independent measurement, not an algebraic rearrangement. It agrees
# with Delta GPE* - Delta N_D to 0.04 TN/m rms, and that difference IS
# the closure residual (1-2 % of the terms, conventions §1.3).
figures_dir.mkdir(exist_ok=True)

# Brand palette (matches presentation/theme.scss)
C_STD, C_WAL            = "#002147", "#E5007D"
C_SLATE, C_RULE, C_BAND = "#404B74", "#BFC3D1", "#F2F3F5"
models = [
    ("STD", C_STD, records_STD),
    ("WAL", C_WAL, records_WAL),
]

fig, ax = plt.subplots(figsize=(9, 5))

for label, color, rec in models:
    if rec is None:
        continue
    t_Myr = rec["t_yr"] / 1e6

    # ΔN_D — bold solid, primary
    ax.plot(t_Myr, rec["delta_Fd_R"] * 1e-12,
            "-o", color=color, lw=2.0, ms=5.5,
            markeredgecolor="white", markeredgewidth=0.6,
            label=fr"{label}  $\Delta N_D$")

    # \Delta\mathrm{GPE}^* — dashed, comparison
    ax.plot(t_Myr, rec["delta_GPE_R"] * 1e-12,
            "--s", color=color, lw=1.7, ms=5.5,
            markeredgecolor="white", markeredgewidth=0.6,
            label=fr"{label}  $\Delta\mathrm{{GPE}}^*$")

    # F_B, as measured and cached -- not ΔGPE* - ΔN_D rearranged
    F_B_R = rec["FB_R"] * 1e-12
    ax.plot(t_Myr, F_B_R,
            ":^", color=color, lw=1.5, ms=5.0,
            markeredgecolor="white", markeredgewidth=0.5,
            label=fr"{label}  $F_B$")

ax.axhline(0, color="k", lw=0.7)
# NO DRIVING/RESISTING HALF-PLANE. With the balance arranged as
# ΔGPE* = ΔN_D + F_B all three terms are positive -- the driving term and
# both resisting ones -- so which half a curve sits in is set by how the
# balance is written, not by the frame, and a shaded band cannot label
# them (conventions §1.1, the 2026-09-30 correction).

ax.set_xlabel("Model time  [Myr]", fontsize=17)
ax.set_ylabel(r"force per unit length   [TN m$^{-1}$]", fontsize=17)
ax.set_xlim(0, 82)
ax.set_ylim(-1, 7)
ax.legend(loc="upper center", frameon=False, fontsize=12, ncol=2)
ax.grid(True, alpha=0.25, color=C_RULE, lw=0.6)
ax.tick_params(labelsize=11)

fig.tight_layout()
fig.savefig(figures_dir / "delta_GPE_R_delta_Fd_R_time_series.png",
            bbox_inches="tight", dpi=220)
plt.show()

### The column-wise resultant

Main-text Figure 5 pairs the plate-wide balance above with $N_D$ at the
three columns. Read together they carry the paper's central negative
result: $\Delta N_D$ generally **resists**, while $N_D(x_T)$ itself is
compression-like in the median state of both runs. A positive $N_D(x_I)$
in STD is generated across the trench side by the trench pull — it is not
a tension-like edge force arriving through the hinge.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.2), sharey=True)
for ax, (key, rec, col) in zip(axes, [("STD", records_STD, "#002147"),
                                      ("WAL", records_WAL, "#E5007D")]):
    t = rec["t_yr"] / 1e6
    ax.axhline(0, color="k", lw=1.2)
    ax.plot(t, rec["Fd_xT"] / 1e12, "-",  color=col, lw=2.0, label="$N_D(x_T)$")
    ax.plot(t, rec["nd_I"]  / 1e12, "--", color=col, lw=1.4, label="$N_D(x_I)$")
    ax.plot(t, rec["nd_R"]  / 1e12, ":",  color=col, lw=1.4, label="$N_D(x_R)$")
    ax.set_title(f"{key}", fontsize=11, loc="left")
    ax.set_xlabel("Model time [Myr]")
    ax.grid(alpha=0.25, color="#BFC3D1", lw=0.6)
    for lab, yy, va in (("TENSION-LIKE", 0.95, "top"),
                        ("COMPRESSION-LIKE", 0.05, "bottom")):
        ax.text(0.5, yy, lab, transform=ax.transAxes, ha="center", va=va,
                fontsize=10, fontweight="bold", color="0.84", zorder=0)
    print(f"{key}: N_D median at x_T {np.median(rec['Fd_xT'])/1e12:+.2f}, "
          f"x_I {np.median(rec['nd_I'])/1e12:+.2f}, "
          f"x_R {np.median(rec['nd_R'])/1e12:+.2f} TN/m; "
          f"tension-like at the trench {100*np.mean(rec['Fd_xT']>0):.0f} % of states")
axes[0].set_ylabel("$N_D$ at a column  [TN/m]")
axes[0].legend(frameon=False, ncol=3, fontsize=9, loc="lower left")
plt.tight_layout(); plt.show()

### What is left once the trends are removed

The three terms are dominated by a secular rise, and at that scale they
read as three smooth, well-separated curves. Removing a linear trend from
each shows the part that is not smooth — and it is neither small nor the
same in the two runs. In STD the residuals of $\Delta N_D$ and $F_B$ are
near mirror images: the two resisting terms exchange with each other while
the drive stays comparatively quiet.

In [ ]:
fit = lambda t, y: np.polyval(np.polyfit(t, y, 1), t)
fig, axes = plt.subplots(1, 2, figsize=(12.0, 3.8), sharey=True)
for ax, (key, rec, col) in zip(axes, [("STD", records_STD, "#002147"),
                                      ("WAL", records_WAL, "#E5007D")]):
    t = rec["t_yr"] / 1e6
    gpe = rec["delta_GPE_R"] / 1e12
    nd  = rec["delta_Fd_R"] / 1e12
    fb  = rec["FB_R"] / 1e12
    ax.axhline(0, color="k", lw=0.7)
    for y, c, lw, lab in ((gpe, "b", 3.0, r"$\Delta$GPE*"),
                          (nd, "k", 1.5, r"$\Delta N_D$"),
                          (fb, "red", 2.0, "$F_B$")):
        ax.plot(t, y - fit(t, y), color=c, lw=lw,
                alpha=0.6 if c == "b" else 1.0, label=lab)
    r = np.corrcoef(nd - fit(t, nd), fb - fit(t, fb))[0, 1]
    ax.set_title(rf"{key}   r(residual $\Delta N_D$, $F_B$) = {r:+.2f}",
                 fontsize=10.5, loc="left")
    ax.set_xlabel("Model time [Myr]")
    ax.grid(alpha=0.25, color="#BFC3D1", lw=0.6)
    print(f"{key}: residual s.d. drive {np.std(gpe-fit(t,gpe)):.3f}, "
          f"dN_D {np.std(nd-fit(t,nd)):.3f}, F_B {np.std(fb-fit(t,fb)):.3f} TN/m; "
          f"r = {r:+.2f}")
axes[0].set_ylabel("residual about\nthe trend [TN/m]")
axes[0].legend(frameon=False, ncol=3, fontsize=9)
plt.tight_layout(); plt.show()

**Interpretation**

* The total $\Delta\mathrm{GPE}^*$ splits into non-overlapping **trench pull** and **ridge push** components sharing the first isostatic column $x_I$ as their common reference.
* Trench pull dominates early (roughly 3× ridge push in the first ~10 Myr); ridge push grows steadily as the plate cools and its isostatic topography builds, and the two become comparable by the end of the run — consistent with plate age at the trench more than doubling over the model duration.

In [ ]:
# Decompose \Delta\mathrm{GPE}^*(x_R, x_T) — the full ridge-to-trench column-GPE difference —
# into its two regimes:
#
#   trench pull   \Delta\mathrm{GPE}^*(x_I, x_T) = \mathrm{GPE}^*(x_I) - \mathrm{GPE}^*(x_T)   (records['delta_GPE'])
#   ridge push    \Delta\mathrm{GPE}^*(x_R, x_I) = \mathrm{GPE}^*(x_R) - \mathrm{GPE}^*(x_I)   (delta_GPE_R - delta_GPE)
#   total         \Delta\mathrm{GPE}^*(x_R, x_T) = \mathrm{GPE}^*(x_R) - \mathrm{GPE}^*(x_T)   (records['delta_GPE_R'])
#
# By construction, total = trench pull + ridge push.
figures_dir.mkdir(exist_ok=True)

# Brand palette
C_STD, C_WAL            = "#002147", "#E5007D"
C_SLATE, C_RULE, C_BAND = "#404B74", "#BFC3D1", "#F2F3F5"
models = [
    ("STD", C_STD, records_STD),
    ("WAL", C_WAL, records_WAL),
]

fig, ax = plt.subplots(figsize=(9, 5))

for label, color, rec in models:
    if rec is None:
        continue
    t_Myr = rec["t_yr"] / 1e6

    dGPE_total = rec["delta_GPE_R"]                       # ridge to trench (total)
    dGPE_TP    = rec["delta_GPE"]                          # isostatic to trench (trench pull)
    dGPE_RP    = rec["delta_GPE_R"] - rec["delta_GPE"]     # ridge to isostatic (ridge push)

    # Trench pull — bold solid
    ax.plot(t_Myr, dGPE_TP * 1e-12,
            "-o", color=color, lw=2.0, ms=5.5,
            markeredgecolor="white", markeredgewidth=0.6,
            label=fr"{label}  trench pull")

    # Ridge push — dashed
    ax.plot(t_Myr, dGPE_RP * 1e-12,
            "--s", color=color, lw=1.7, ms=5.5,
            markeredgecolor="white", markeredgewidth=0.6,
            label=fr"{label}  ridge push")

    # Total — dotted, lighter
    #ax.plot(t_Myr, dGPE_total * 1e-12,
    #        ":^", color=color, lw=1.5, ms=5.0,
    #        markeredgecolor="white", markeredgewidth=0.5,
    #        label=fr"{label}  total")

ax.axhline(0, color="k", lw=0.7)
ax.axhspan(-100, 0, color=C_BAND, zorder=-2)
ax.text(0.012, 0.97, "driving", transform=ax.transAxes,
        ha="left", va="top", fontsize=17, color=C_SLATE, style="italic")
#ax.text(0.012, 0.03, "resisting", transform=ax.transAxes,
#        ha="left", va="bottom", fontsize=17, color=C_SLATE, style="italic")

ax.set_xlabel("Model time  [Myr]", fontsize=17)
ax.set_ylabel(r"$\Delta\mathrm{GPE}^*$   [TN m$^{-1}$]", fontsize=17)
ax.set_xlim(0, 82)
ax.set_ylim(0, 4)
ax.legend(loc="upper center", frameon=False, fontsize=14, ncol=2)
ax.grid(True, alpha=0.25, color=C_RULE, lw=0.6)
ax.tick_params(labelsize=11)

fig.tight_layout()
fig.savefig(figures_dir / "GPE_decomposition_time_series.png",
            bbox_inches="tight", dpi=220)
plt.show()


### The driving-force partition with the back-tilt force

Per `ANALYSIS_CONVENTIONS.md` §6b (single-step anatomy in
`further_analysis/fluidity_back_tilt.ipynb`): trench pull
$\Delta\mathrm{GPE}^*(x_I, x_T)$, measured ridge push
$\Delta\mathrm{GPE}^*(x_R, x_I)$, and the **back-tilt force**
$F_{tilt} = -q_c(x_R)\,z_c$ — the ridge push *suppressed* by the
asthenospheric return-flow gradient tilting the plate down toward the ridge
($q_c$: deep σ_zz residual rel. the $x_I$ column, 120–190 km band, 100-km
smoothed). It can move over to measured ridge push as the channel evolves;
isostatic ridge push = measured + $F_{tilt}$. **STD vs WAL is the controlled
experiment**: only the weak asthenospheric layer differs.

In [ ]:
# Partition through time, one panel per model (STD | WAL), shared y-axis.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, (label, rec) in zip(axes, [("STD", records_STD), ("WAL", records_WAL)]):
    if rec is None:
        ax.set_visible(False)
        continue
    tM = rec["t_yr"] / 1e6
    tp = rec["delta_GPE"]                       # trench pull (x_I - x_T)
    rp = rec["delta_GPE_R"] - rec["delta_GPE"]  # measured ridge push (x_R - x_I)
    ft = rec["f_tilt"]
    ax.plot(tM, tp * 1e-12, ".-", color="orange", lw=1.6,
            label=r"trench pull $\Delta\mathrm{GPE}^*$ ($x_I - x_T$)")
    ax.plot(tM, rp * 1e-12, ".-", color="#002147", lw=1.6,
            label=r"ridge push (measured, $x_R - x_I$)")
    ax.plot(tM, ft * 1e-12, ".-", color="#E5007D", lw=1.6,
            label=r"back-tilt $F_{tilt} = -q_c(x_R)\,z_c$")
    ax.plot(tM, (rp + ft) * 1e-12, "--", color="0.55", lw=1.2,
            label="ridge push (isostatic = measured + $F_{tilt}$)")
    ax.axhline(0, color="0.6", lw=0.5)
    ax.set_xlabel("t [Myr]")
    ax.set_title(label)
    ax.legend(fontsize=8)
    print(f"{label}: run-median  trench pull {np.median(tp)*1e-12:+.2f}, "
          f"RP meas {np.median(rp)*1e-12:+.2f}, "
          f"F_tilt {np.median(ft)*1e-12:+.2f} TN/m, "
          f"suppression {np.median(ft / (rp + ft)):.2f}")

axes[0].set_ylabel("Force per unit distance [TN/m]")
fig.suptitle("Fluidity — trench pull / ridge push / back-tilt through time", fontsize=12)
fig.tight_layout()
out = figures_dir / "partition_back_tilt_time_series.png"
fig.savefig(out, dpi=160, bbox_inches="tight")
print("figure:", out)

### Why does the back-tilt change? The channel-flow test

If the return-flow pressure gradient obeys $\partial p/\partial x \sim \eta U / h^2$,
$F_{tilt}$ should track plate speed; channel thickening/warming would instead decay it
even at constant speed. The discriminating statistic is the correlation of
**step-to-step increments**, which is immune to shared secular trends. Verdict here:
the back-tilt is **velocity-following** — the $F_{tilt}$ series is close to a shifted
copy of $|v_{sp}|$, oscillation for oscillation, and WAL needs less pressure gradient
per unit speed (the weak layer). In dynamic-topography terms the ridge-end depression
$\Delta e = -q_c/\rho g$ is hundreds of metres, appearing and fading with plate speed
on ~10-Myr timescales.

In [ ]:
# F_tilt vs plate speed: twin time series + scatter, per model.
fig, axes = plt.subplots(2, 2, figsize=(13, 8.6))

for j, (label, rec, col) in enumerate([("STD", records_STD, "#002147"),
                                       ("WAL", records_WAL, "#E5007D")]):
    if rec is None:
        continue
    tM = rec["t_yr"] / 1e6
    u = np.abs(rec["v_sp"])            # numerically cm/yr (see loop note)
    ft = rec["f_tilt"] * 1e-12
    r_lvl = np.corrcoef(u, ft)[0, 1]
    r_inc = np.corrcoef(np.diff(u), np.diff(ft))[0, 1]
    print(f"{label}: corr(|v_sp|, F_tilt) = {r_lvl:+.2f}   increments {r_inc:+.2f}   "
          f"slope {np.polyfit(u, ft, 1)[0]:.3f} TN/m per cm/yr")

    ax = axes[0, j]
    ax.plot(tM, u, ".-", color=col, lw=1.4)
    ax.set_ylabel("$|v_{sp}|$ [cm/yr]", color=col)
    ax.set_xlabel("t [Myr]")
    ax.set_title(f"{label}   r = {r_lvl:+.2f} (increments {r_inc:+.2f})")
    ax2 = ax.twinx()
    ax2.plot(tM, ft, ".--", color="0.45", lw=1.2)
    ax2.set_ylabel("$F_{tilt}$ [TN/m]", color="0.35")
    ax2.tick_params(axis="y", colors="0.35")

    ax = axes[1, j]
    sc = ax.scatter(u, ft, c=tM, cmap="viridis", s=30)
    ax.set_xlabel("$|v_{sp}|$ [cm/yr]")
    ax.set_ylabel("$F_{tilt}$ [TN/m]")
    plt.colorbar(sc, ax=ax, label="t [Myr]")

fig.suptitle("Back-tilt force vs subducting-plate speed — the channel-flow test", fontsize=12)
fig.tight_layout()
out = figures_dir / "back_tilt_velocity_correlation.png"
fig.savefig(out, dpi=160, bbox_inches="tight")
print("figure:", out)

In [ ]:
# V(x_T) time-series for STD and WAL — vertical shear stress resultant
# integrated through the column at the trench, plotted through model time.
figures_dir.mkdir(exist_ok=True)

# Brand model palette (single-quantity-per-model plot, so colour = model)
C_STD, C_WAL            = "#002147", "#E5007D"
C_SLATE, C_RULE, C_BAND = "#404B74", "#BFC3D1", "#F2F3F5"
models = [
    ("STD", C_STD, records_STD),
    ("WAL", C_WAL, records_WAL),
]

fig, ax = plt.subplots(figsize=(9, 5))

for label, color, rec in models:
    if rec is None:
        continue
    t_Myr = rec["t_yr"] / 1e6
    ax.plot(t_Myr, -1*rec["V_xT"] * 1e-12,
            "-o", color=color, lw=2.0, ms=5.5,
            markeredgecolor="white", markeredgewidth=0.6,
            label=label)

ax.axhline(0, color="k", lw=0.7)

ax.set_xlabel("Model time  [Myr]", fontsize=17)
ax.set_ylabel(r"$V(x_T)$   [TN m$^{-1}$]", fontsize=17)
ax.set_xlim(0, 82)
ax.legend(loc="best", frameon=False, fontsize=14)
ax.grid(True, alpha=0.25, color=C_RULE, lw=0.6)
ax.tick_params(labelsize=11)

fig.tight_layout()
fig.savefig(figures_dir / "V_xT_time_series.png",
            bbox_inches="tight", dpi=220)
plt.show()


---

Both notebooks stop here. The published figures — the ones with the
captions, the shared colour scheme and the tables of quoted numbers — are
produced by `scripts/fig_*.py`, one script per figure. This notebook
exists so the analysis behind them can be taken apart and changed.